# Exploratory notebook template

Use this notebook in `scratch/notebooks/` to pull in artifacts from `tasks/.../output` and explore them.

A few ground rules for this repo:
- Read stable inputs from task outputs, or from downstream `input/` symlinks when you want to inspect exactly what a task consumes.
- Treat work here as exploratory. If a table, figure, or dataset becomes part of the workflow, move it into a proper task under `tasks/`.
- Keep paths relative to the repo structure rather than hardcoding machine-specific paths.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Find the repo root and load parameters

This assumes the notebook kernel starts somewhere inside the repository. The loop walks up until it finds `config/params.yaml`.


In [ ]:
repo_root = Path.cwd().resolve()

while not (repo_root / "config" / "params.yaml").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not find repo root. Start Jupyter from somewhere inside this repo.")
    repo_root = repo_root.parent

scratch_dir = repo_root / "scratch" / "notebooks"
config_path = repo_root / "config" / "params.yaml"

print(f"repo_root: {repo_root}")
print(f"scratch_dir: {scratch_dir}")
print(f"config_path: {config_path}")


In [ ]:
with open(config_path, "r", encoding="utf-8") as handle:
    params = yaml.safe_load(handle)

params


## Point to the artifact you want to explore

Replace `dataset_relpath` with the task output you want. The example below points at a real artifact in this repo.

If you want to inspect the exact file a downstream task sees, you can instead point at something like `tasks/<task>/input/<file>`.


In [ ]:
dataset_relpath = Path("tasks/data/prep_compustat/output/firms_quarterly.parquet")
dataset_path = repo_root / dataset_relpath

if not dataset_path.exists():
    raise FileNotFoundError(dataset_path)

dataset_path


## Load the data

Add another branch if you regularly work with another file type.


In [ ]:
if dataset_path.suffix == ".parquet":
    df = pd.read_parquet(dataset_path)
elif dataset_path.suffix == ".feather":
    df = pd.read_feather(dataset_path)
elif dataset_path.suffix == ".dta":
    df = pd.read_stata(dataset_path)
elif dataset_path.suffix == ".pkl":
    df = pd.read_pickle(dataset_path)
elif dataset_path.suffix == ".csv" or dataset_path.name.endswith(".csv.gz"):
    df = pd.read_csv(dataset_path)
else:
    raise ValueError(f"Add a reader for {dataset_path.name}")

print(df.shape)
df.head()


## First-pass inspection

These are the basic checks I usually want immediately.


In [ ]:
df.info()


In [ ]:
pd.Series(df.columns, name="column").to_frame().head(50)


In [ ]:
missing_share = df.isna().mean().sort_values(ascending=False)
missing_share.head(25)


## Make a working copy and start exploring

Keep the raw loaded object untouched when possible.


In [ ]:
work = df.copy()

# Examples:
# work = work.loc[work["fyearq"].between(2000, 2020)].copy()
# work = work.loc[work["sic"].between(2000, 3999)].copy()
# work["leverage"] = (work["dlttq"].fillna(0) + work["dlcq"].fillna(0)) / work["atq"]
# work["log_atq"] = np.log(work["atq"])

work.head()


In [ ]:
summary_cols = [col for col in ["atq", "saleq", "capxy"] if col in work.columns]

if summary_cols:
    work[summary_cols].describe().T
else:
    print("Set summary_cols to variables that exist in this dataset.")


In [ ]:
plot_col = "atq"

if plot_col in work.columns:
    work[plot_col].dropna().hist(bins=50, figsize=(7, 4))
    plt.title(plot_col)
    plt.show()
else:
    print(f"Set plot_col to a column in this dataset. {plot_col!r} is not present.")


## When the notebook stops being exploratory

Once you know you need a stable figure, table, cleaned sample, or derived dataset, move that logic into a task under `tasks/` so it becomes reproducible and available downstream.
